# Coffee Roasting Neural Network — Standalone Version

This is a **self-contained** version of `C2_W1_Lab02_CoffeeRoasting_TF`.

The original lab imports helper functions from two files that DeepLearning.AI
supplies (`lab_utils_common.py`, `lab_coffee_utils.py`) which you don't have.
So this notebook does two things differently:

1. It **recreates the coffee-roasting dataset generator and the plotting
   helpers** itself, using plain NumPy + Matplotlib, so every cell runs with
   no missing files.
2. Every cell has a markdown explanation above it that walks through the
   **Python and TensorFlow syntax**, not just what the code produces.

Run the cells top to bottom.


## 1. Imports

- `numpy as np` → array math.
- `matplotlib.pyplot as plt` → plotting.
- `tensorflow as tf` and the `Sequential` / `Dense` classes → build the network.
- The `logging` lines just silence TensorFlow's noisy warning messages; they
  don't affect the model.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

np.set_printoptions(precision=2)


## 2. Recreating the dataset

The real lab calls `load_coffee_data()` from a hidden file. Here is a
functionally equivalent version, written from scratch, so you can see
exactly how the data is generated instead of trusting a black box.

**Logic:**
- Pick random `(temperature, duration)` pairs.
- `temperature` is scaled into roughly `150–285 °C`.
- `duration` is scaled into roughly `11.5–15.5` minutes.
- A point is labeled a **"good roast" (`y=1`)** only if it falls inside a
  triangular region: temperature between 175–260 °C, duration between 12–15
  minutes, **and** below a diagonal line (because as temperature rises, the
  ideal duration should shrink — you don't want to overcook a hot roast).
- Everything else is a **"bad roast" (`y=0`)**.

### Python syntax notes
- `rng = np.random.default_rng(2)` creates a random number generator with a
  fixed **seed** (`2`), so you get the same "random" data every time you
  rerun the notebook. This is for reproducibility.
- `rng.random(400).reshape(-1, 2)` makes 400 random numbers between 0 and 1,
  then reshapes them into `(200, 2)` — 200 rows (examples), 2 columns
  (features: temperature, duration).
- `X[:, 1]` selects **all rows, column 1** (duration). `X[:, 0]` selects
  column 0 (temperature). This is NumPy's slicing syntax.
- The `for t, d in X:` loop **unpacks each row** into `t` (temperature) and
  `d` (duration) as it iterates.


In [ ]:
def load_coffee_data():
    '''Creates a synthetic coffee-roasting dataset.
    Good roast: 12-15 min duration AND 175-260 C, below a diagonal cutoff.
    '''
    rng = np.random.default_rng(2)
    X = rng.random(400).reshape(-1, 2)
    X[:, 1] = X[:, 1] * 4 + 11.5        # duration -> ~11.5 to 15.5 minutes
    X[:, 0] = X[:, 0] * (285 - 150) + 150   # temperature -> ~150 to 285 C
    Y = np.zeros(len(X))

    i = 0
    for t, d in X:
        y = -3 / (260 - 175) * t + 21   # diagonal cutoff line
        if (t > 175 and t < 260 and d > 12 and d < 15 and d <= y):
            Y[i] = 1
        else:
            Y[i] = 0
        i += 1

    return (X, Y.reshape(-1, 1))


X, Y = load_coffee_data()
print(X.shape, Y.shape)


## 3. Plotting the raw data (`plt_roast` rebuilt)

- `Y == 1` produces a boolean array (`True`/`False` per row).
- `X[Y[:, 0] == 1, 0]` is **boolean indexing**: keep only the rows where the
  label is 1, then take column 0 (temperature) from those rows.
- `facecolors='none'` draws hollow circles for the "bad roast" points.


In [ ]:
def plt_roast(X, Y):
    Y = Y.reshape(-1,)
    pos = Y == 1
    neg = Y == 0

    fig, ax = plt.subplots(1, 1, figsize=(6, 5))
    ax.scatter(X[pos, 0], X[pos, 1], marker='x', s=80, c='red', label="Good Roast")
    ax.scatter(X[neg, 0], X[neg, 1], marker='o', s=80, facecolors='none',
               edgecolors='blue', label="Bad Roast")

    ax.set_xlabel("Temperature (Celsius)")
    ax.set_ylabel("Duration (minutes)")
    ax.set_title("Coffee Roasting")
    ax.legend()
    plt.show()

plt_roast(X, Y)


## 4. Normalizing the data

Neural networks train faster and more reliably when input features are on a
similar scale. Temperature ranges over ~150 units while duration ranges over
only ~4 units — very different scales — so we normalize both to have
**mean 0 and standard deviation 1** (this is called standardization).

### Syntax
- `tf.keras.layers.Normalization(axis=-1)` creates a special Keras layer
  whose only job is to normalize data. `axis=-1` means "normalize each
  feature (last axis) independently," so temperature and duration each get
  their own mean/variance.
- `norm_l.adapt(X)` is **not** training. It just scans `X` once to compute
  the mean and variance of each column and stores them inside the layer.
- `norm_l(X)` then applies `(x - mean) / std` to every value, using the
  numbers it just learned.
- This is **not part of the model** you'll build below — it's a
  preprocessing step you apply before and after using the model.


In [ ]:
print(f"Temperature Max, Min pre normalization:  {np.max(X[:,0]):0.2f}, {np.min(X[:,0]):0.2f}")
print(f"Duration    Max, Min pre normalization:  {np.max(X[:,1]):0.2f}, {np.min(X[:,1]):0.2f}")

norm_l = tf.keras.layers.Normalization(axis=-1)
norm_l.adapt(X)          # learns mean, variance from X
Xn = norm_l(X)            # applies (x - mean) / std

print(f"Temperature Max, Min post normalization: {np.max(Xn[:,0]):0.2f}, {np.min(Xn[:,0]):0.2f}")
print(f"Duration    Max, Min post normalization: {np.max(Xn[:,1]):0.2f}, {np.min(Xn[:,1]):0.2f}")


## 5. Expanding the dataset (`np.tile`)

`np.tile(Xn, (1000, 1))` copies the entire `Xn` array **1000 times, stacked
vertically** (the `(1000, 1)` means "repeat 1000× along rows, 1× along
columns"). This is a shortcut so gradient descent sees more examples per
epoch without needing new real data — purely to make training more stable
for the lab.

**Note:** training on 200,000 rows for 10 epochs can take a few minutes on a
laptop CPU. If it feels slow, change `1000` to something smaller like `50`
in the cell below.


In [ ]:
Xt = np.tile(Xn, (1000, 1))
Yt = np.tile(Y, (1000, 1))
print(Xt.shape, Yt.shape)


## 6. Building the network

```python
model = Sequential([
    tf.keras.Input(shape=(2,)),
    Dense(3, activation='sigmoid', name='layer1'),
    Dense(1, activation='sigmoid', name='layer2')
])
```

- `Sequential([...])` stacks layers **in the order given** — the output of
  one layer becomes the input of the next.
- `tf.keras.Input(shape=(2,))` isn't a real computing layer. It just tells
  Keras "expect 2 features per example," so it can size the weight matrices
  immediately instead of waiting for the first real data (recall in Lab 1
  the weights only appeared after the first call — this avoids that).
- `Dense(3, activation='sigmoid', name='layer1')`: **3 neurons**, each doing
  `sigmoid(w·x + b)`. Since the input has 2 features and there are 3 units,
  this layer's weight matrix `W1` has shape `(2, 3)` and bias `b1` has shape
  `(3,)`.
- `Dense(1, activation='sigmoid', name='layer2')`: **1 neuron** taking the 3
  outputs of layer 1 as its input. So `W2` is `(3, 1)` and `b2` is `(1,)`.
- `name='layer1'` lets you fetch that specific layer later with
  `model.get_layer('layer1')`.
- `tf.random.set_seed(1234)` fixes TensorFlow's random number generator so
  the random initial weights are the same every run.


In [ ]:
tf.random.set_seed(1234)

model = Sequential([
    tf.keras.Input(shape=(2,)),
    Dense(3, activation='sigmoid', name='layer1'),
    Dense(1, activation='sigmoid', name='layer2')
])

model.summary()


## 7. Checking parameter counts and shapes

`model.get_layer("layer1").get_weights()` returns `[W1, b1]` for that layer,
same pattern as Lab 1's `get_weights()` / `set_weights()`.

- Layer 1: input has 2 features, 3 units → `W1` is `(2, 3)` = 6 weights,
  plus 3 biases = **9 parameters**.
- Layer 2: input has 3 features (the 3 outputs of layer 1), 1 unit →
  `W2` is `(3, 1)` = 3 weights, plus 1 bias = **4 parameters**.

This matches the `model.summary()` table above.


In [ ]:
W1, b1 = model.get_layer("layer1").get_weights()
W2, b2 = model.get_layer("layer2").get_weights()

print(f"W1{W1.shape}:\n", W1, f"\nb1{b1.shape}:", b1)
print(f"W2{W2.shape}:\n", W2, f"\nb2{b2.shape}:", b2)


## 8. Compiling and training

```python
model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
)
model.fit(Xt, Yt, epochs=10)
```

- `model.compile(...)` doesn't train anything — it just configures **how**
  training will happen:
  - `loss=BinaryCrossentropy()` — the error measure used for yes/no
    classification problems (appropriate since our labels are 0 or 1 and
    the output is a sigmoid probability).
  - `optimizer=Adam(learning_rate=0.01)` — the algorithm that updates the
    weights each step, and how big each update step is.
- `model.fit(Xt, Yt, epochs=10)` is where training actually happens
  (gradient descent, covered in detail in Week 2). `epochs=10` means the
  entire dataset is passed through 10 times.
- The training log line like `6250/6250 [====] - loss: 0.1782` means: there
  are 200,000 examples split into **batches** of 32 (TensorFlow's default
  batch size), so `200000 / 32 = 6250` batches per epoch, and `loss`
  is the current error (lower is better).


In [ ]:
model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
)

model.fit(
    Xt, Yt,
    epochs=10,
)


## 9. Weights after training

Same `get_weights()` call as before — but now the numbers have changed,
because `fit()` adjusted them to minimize the loss.


In [ ]:
W1, b1 = model.get_layer("layer1").get_weights()
W2, b2 = model.get_layer("layer2").get_weights()
print("W1:\n", W1, "\nb1:", b1)
print("W2:\n", W2, "\nb2:", b2)


## 10. (Optional) Load known-good weights

Because your training run's random data/seed may differ slightly from the
course's, here are weights known to work well, taken from the original
lab. Run this cell if you want your predictions and plots below to
definitely match the lab's discussion — otherwise skip it and use your own
trained weights.

`set_weights([W, b])` overwrites a layer's weights with the arrays you give
it, same pattern as `linear_layer.set_weights(...)` in Lab 1.


In [ ]:
W1 = np.array([
    [-8.94,  0.29, 12.89],
    [-0.17, -7.34, 10.79]])
b1 = np.array([-9.87, -9.28, 1.01])

W2 = np.array([
    [-31.38],
    [-27.86],
    [-32.79]])
b2 = np.array([15.54])

model.get_layer("layer1").set_weights([W1, b1])
model.get_layer("layer2").set_weights([W2, b2])

# confirm
W1, b1 = model.get_layer("layer1").get_weights()
W2, b2 = model.get_layer("layer2").get_weights()
print("W1:\n", W1, "\nb1:", b1)
print("W2:\n", W2, "\nb2:", b2)


## 11. Making predictions

- `X_test` is two new, unseen examples: one that should roast well, one
  that shouldn't (17 minutes is far too long).
- **Important:** since the model was trained on *normalized* data, any new
  data must be normalized with the **same** `norm_l` layer (which already
  learned the training data's mean/variance) before predicting.
- `model.predict(...)` runs the whole network and returns a probability
  between 0 and 1 for each row — the model's estimate of "chance this is a
  good roast."


In [ ]:
X_test = np.array([
    [200, 13.9],   # expected: positive / good roast
    [200, 17]])    # expected: negative / bad roast (too long)

X_testn = norm_l(X_test)
predictions = model.predict(X_testn)
print("predictions = \n", predictions)


## 12. Turning probabilities into decisions

A probability isn't a yes/no answer by itself, so we apply a **threshold**:
anything `>= 0.5` becomes a "good roast" decision (1), anything below
becomes "bad roast" (0).

Two equivalent ways to write this:

```python
# explicit loop version
yhat = np.zeros_like(predictions)
for i in range(len(predictions)):
    if predictions[i] >= 0.5:
        yhat[i] = 1
    else:
        yhat[i] = 0
```

```python
# vectorized, one-line version
yhat = (predictions >= 0.5).astype(int)
```

`predictions >= 0.5` compares every element at once and returns a boolean
array. `.astype(int)` converts `True/False` into `1/0`. This vectorized
style avoids writing an explicit Python loop and is the more idiomatic
NumPy way to do it.


In [ ]:
yhat = (predictions >= 0.5).astype(int)
print(f"decisions = \n{yhat}")


## 13. Visualizing what each hidden unit learned

This rebuilds the idea behind the lab's `plt_layer` function: for every
point on a grid of (temperature, duration) values, compute what each of the
3 layer-1 neurons outputs, and shade the plot by that value.

### Syntax notes
- `np.meshgrid` builds a grid of `(temp, duration)` coordinate pairs
  covering the whole plot area, so we can evaluate the layer everywhere,
  not just at the original data points.
- `np.c_[a.ravel(), b.ravel()]` flattens the two grids and stacks them as
  columns, producing a `(N, 2)` array — the same shape the model expects.
- The sigmoid formula is applied manually with NumPy (`1 / (1 + np.exp(-z))`)
  to compute each hidden unit's output directly from `W1`, `b1`, without
  calling the Keras layer — same idea as comparing `layer(x)` to
  `np.dot(w, x) + b` in Lab 1.


In [ ]:
def sigmoid_np(z):
    return 1 / (1 + np.exp(-z))

def plt_layer(X, Y, W1, b1, norm_l):
    Y = Y.reshape(-1,)
    t_range = np.linspace(X[:, 0].min() - 5, X[:, 0].max() + 5, 100)
    d_range = np.linspace(X[:, 1].min() - 0.3, X[:, 1].max() + 0.3, 100)
    TT, DD = np.meshgrid(t_range, d_range)
    grid = np.c_[TT.ravel(), DD.ravel()]
    grid_n = norm_l(grid).numpy()

    Z_all = sigmoid_np(grid_n @ W1 + b1)   # shape (10000, 3) -> one column per unit

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    pos = Y == 1
    neg = Y == 0

    for unit in range(3):
        ax = axes[unit]
        Z = Z_all[:, unit].reshape(TT.shape)
        cs = ax.contourf(TT, DD, Z, levels=20, cmap='Blues')
        ax.scatter(X[pos, 0], X[pos, 1], marker='x', s=40, c='red')
        ax.scatter(X[neg, 0], X[neg, 1], marker='o', s=40, facecolors='none', edgecolors='blue')
        ax.set_title(f"Layer 1, unit {unit}")
        ax.set_xlabel("Temperature (Celsius)")
        ax.set_ylabel("Duration (minutes)")
        fig.colorbar(cs, ax=ax)

    plt.tight_layout()
    plt.show()

plt_layer(X, Y, W1, b1, norm_l)


### What you should see

Each of the 3 subplots shades differently: one unit lights up mostly for
*low temperature*, another for *short duration*, and the third for *bad
combinations of time and temperature*. Nobody told the network to divide
the problem this way — **gradient descent found these patterns on its own**
while minimizing the loss. That's the core idea of the whole lab: a hidden
layer of neurons each learn to detect a useful sub-pattern, and the final
layer combines them into one decision.


## 14. Visualizing the whole network's decision

This rebuilds `plt_network`: for every point on the grid, run the **full**
model (both layers) to get a probability, then also show the thresholded
decision.

- `netf = lambda x: model.predict(norm_l(x))` defines a small anonymous
  function with `lambda`. It's shorthand for:
  ```python
  def netf(x):
      return model.predict(norm_l(x))
  ```
  `lambda` is used here because we just need a quick one-line function to
  pass around, not a fully named one.


In [ ]:
def plt_network(X, Y, netf):
    Y = Y.reshape(-1,)
    t_range = np.linspace(X[:, 0].min() - 5, X[:, 0].max() + 5, 60)
    d_range = np.linspace(X[:, 1].min() - 0.3, X[:, 1].max() + 0.3, 60)
    TT, DD = np.meshgrid(t_range, d_range)
    grid = np.c_[TT.ravel(), DD.ravel()]

    probs = netf(grid).reshape(TT.shape)
    decisions = (probs >= 0.5).astype(int)

    pos = Y == 1
    neg = Y == 0

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    cs = axes[0].contourf(TT, DD, probs, levels=20, cmap='Blues')
    axes[0].scatter(X[pos, 0], X[pos, 1], marker='x', s=40, c='red', label="Good Roast")
    axes[0].scatter(X[neg, 0], X[neg, 1], marker='o', s=40, facecolors='none', edgecolors='blue', label="Bad Roast")
    axes[0].set_title("Network probability")
    axes[0].set_xlabel("Temperature (Celsius)")
    axes[0].set_ylabel("Duration (minutes)")
    axes[0].legend()
    fig.colorbar(cs, ax=axes[0])

    axes[1].contourf(TT, DD, decisions, levels=1, cmap='Oranges', alpha=0.5)
    axes[1].scatter(X[pos, 0], X[pos, 1], marker='x', s=40, c='red', label="Good Roast")
    axes[1].scatter(X[neg, 0], X[neg, 1], marker='o', s=40, facecolors='none', edgecolors='blue', label="Bad Roast")
    axes[1].set_title("Network decision")
    axes[1].set_xlabel("Temperature (Celsius)")
    axes[1].set_ylabel("Duration (minutes)")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


netf = lambda x: model.predict(norm_l(x), verbose=0)
plt_network(X, Y, netf)


## Summary

| Step | What happens |
|---|---|
| `load_coffee_data()` | Builds a synthetic labeled dataset |
| `Normalization` layer | Rescales features to mean 0, std 1 |
| `Sequential([Dense(3, sigmoid), Dense(1, sigmoid)])` | 2-layer network: 3 hidden neurons → 1 output neuron |
| `model.compile(...)` | Sets the loss function and optimizer |
| `model.fit(Xt, Yt, epochs=10)` | Runs gradient descent to learn `W1, b1, W2, b2` |
| `model.predict(...)` | Produces a probability for new inputs |
| `probs >= 0.5` | Converts probability into a yes/no decision |

The key idea: each of the 3 hidden neurons independently learns to detect
one *reason* a roast might be bad (too cold, too short, or a bad time/temp
combo), and the final neuron combines their signals into a single
good/bad decision — exactly what a person doing this by hand would
reason through, but learned automatically from data.
